[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jiehou-lab/urban-ai/blob/main/notebooks/lab0_ai_in_the_browser.ipynb)

# Lab 0: AI in the Browser + First Colab

**Duration:** ~1.25 hours
**TA lead:** Sean
**Course:** Urban AI — AI-Driven Decision Support for Real-World Urban Challenges (MSU AI-Ready Initiative)

## Learning goals
- Load and inspect a real-world-style urban dataset (311 service requests) in Python.
- Summarize service-request patterns with simple charts.
- Practice writing an AI prompt, critiquing the AI's response, and logging the interaction.
- Produce your first AI Use Log entry.


## Before you start: Track A vs. Track B

Every Urban AI lab has two tracks. Both produce the **same artifact**: **AI Use Log entry #1**.

- **Track A — No code (default).** Use a chat-based AI tool to prompt, critique, and log one urban question. No installation, no Python required — use this track if you would rather click through a web tool.
- **Track B — Colab (this notebook).** Run a 10-cell tour: load a city dataset, plot it, and get a (simulated) AI summary of it. You will run pre-written cells and change only the parameters marked `# ▶ CHANGE ME`. You will never need to write code from scratch.

Both tracks end with the same 4 reflection prompts (the last cell of this notebook).


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Fixed seed so everyone in class gets the exact same numbers
RNG = np.random.default_rng(42)

plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

print("Setup complete. Libraries loaded.")

## 2. Load the data: a city's 311 service requests

> **Synthetic-but-realistic data.** The dataset below is generated in this notebook with a fixed
> random seed so the lab runs the same way for everyone, completely offline. It is built to look and
> behave like real urban data, but it is not real. To swap in real data for your own city, instructors
> can replace the data-generation cell with a download/load from a real source such as:
- Your city's Open Data portal (search "[city name] open data 311")
- NYC Open Data "311 Service Requests" dataset (data.cityofnewyork.us)
- data.gov 311 / service-request datasets for other US cities


In [ ]:
n = 600
neighborhoods = ["Riverside", "Old Town", "Eastgate", "Millbrook", "Southside", "Uptown"]
categories = ["Pothole", "Streetlight Outage", "Trash/Recycling", "Noise Complaint",
              "Graffiti", "Water Leak", "Tree/Branch", "Sidewalk Repair"]

start_date = datetime(2025, 1, 1)
dates = [start_date + timedelta(days=int(d)) for d in RNG.integers(0, 180, size=n)]

# Neighborhoods submit requests at different volumes, and get different response times
# (this mirrors real cities, where service-delivery equity varies by neighborhood).
neighborhood_weights = {"Riverside": 0.10, "Old Town": 0.30, "Eastgate": 0.15,
                         "Millbrook": 0.10, "Southside": 0.25, "Uptown": 0.10}
neighborhood_delay_bias = {"Riverside": 0, "Old Town": -1, "Eastgate": 2,
                            "Millbrook": 4, "Southside": 6, "Uptown": -2}

nb_choices = RNG.choice(list(neighborhood_weights.keys()), size=n, p=list(neighborhood_weights.values()))
cat_choices = RNG.choice(categories, size=n)
base_days = RNG.gamma(shape=2.0, scale=2.5, size=n)
days_to_close = np.clip(base_days + [neighborhood_delay_bias[nb] for nb in nb_choices], 0.5, None).round(1)

df311 = pd.DataFrame({
    "request_id": [f"SR-{1000+i}" for i in range(n)],
    "date": dates,
    "neighborhood": nb_choices,
    "category": cat_choices,
    "days_to_close": days_to_close,
})
df311.head()

### What are the most common complaints?
A planner scanning 311 data first wants to know which service categories drive the most demand — that shapes budget and staffing decisions.

In [ ]:
category_counts = df311["category"].value_counts()

fig_category, ax = plt.subplots(figsize=(8, 5))
category_counts.plot(kind="bar", ax=ax)
ax.set_title("311 Service Requests by Category")
ax.set_xlabel("Request category")
ax.set_ylabel("Number of requests")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Which neighborhoods report the most?
Request volume by neighborhood can reflect real problems, but it can also reflect who has time, trust, or ability to report — not necessarily where the problems are worst.

In [ ]:
neighborhood_counts = df311["neighborhood"].value_counts()

fig_neighborhood, ax = plt.subplots(figsize=(8, 5))
neighborhood_counts.plot(kind="bar", ax=ax, color="tab:purple")
ax.set_title("311 Service Requests by Neighborhood")
ax.set_xlabel("Neighborhood")
ax.set_ylabel("Number of requests")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Who waits the longest for a response?
Count of requests only tells half the story. Average time-to-close by neighborhood reveals service-delivery equity — a key thing a planner (and a human reviewer of any AI summary) should check.

In [ ]:
avg_delay = df311.groupby("neighborhood")["days_to_close"].mean().sort_values(ascending=False)

fig_delay, ax = plt.subplots(figsize=(8, 5))
avg_delay.plot(kind="bar", ax=ax, color="tab:orange")
ax.set_title("Average Days to Close 311 Requests, by Neighborhood")
ax.set_xlabel("Neighborhood")
ax.set_ylabel("Average days to close")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(avg_delay)

### Ask an AI to summarize the data
In Track A you would paste a prompt like the one below into an approved chat-AI tool and get back a real,
non-deterministic answer. So this notebook can run offline for everyone, the cell below **simulates**
what that AI response might look like using a template function — but the habit you are practicing is the
same one you will use with a real tool: read the AI's output critically before trusting it.

In [ ]:
def simulated_ai_summary(data):
    # Stand-in for a real chat-AI tool's response (offline-safe). A real tool's wording will differ
    # every time -- always read the actual output, do not assume it says exactly this.
    top_cat = data["category"].value_counts().idxmax()
    top_nb = data["neighborhood"].value_counts().idxmax()
    slow_nb = data.groupby("neighborhood")["days_to_close"].mean().idxmax()
    slow_days = data.groupby("neighborhood")["days_to_close"].mean().max()
    summary = (
        f"Between {data['date'].min().date()} and {data['date'].max().date()}, the city received "
        f"{len(data)} 311 requests. The most common issue was '{top_cat}'. "
        f"'{top_nb}' submitted the most requests overall. "
        f"'{slow_nb}' had the slowest average response time at {slow_days:.1f} days."
    )
    return summary

prompt = "Summarize this 311 dataset in 3 sentences for a city planner."
ai_response = simulated_ai_summary(df311)
print("PROMPT SENT TO AI TOOL:\n", prompt)
print("\nAI RESPONSE:\n", ai_response)

**Critique it.** Notice what the AI response leaves out: it never mentions which neighborhood submits the
*fewest* requests relative to how many people live there, and it treats "most requests" as if it obviously
means "most in need" — it might just mean "most likely to report." This is exactly the kind of gap a human
reviewer has to catch.

## Experiment

Try changing the parameters marked `# ▶ CHANGE ME` in the next cell(s) and re-run. Specifically, try:

1. Change `FOCUS_NEIGHBORHOOD` to a different neighborhood and see which categories dominate there.
2. Raise `MIN_REQUESTS` to see only the most significant categories.
3. Try running the AI-summary cell above after changing the neighborhood weights — does the AI's story change?


In [ ]:
# ▶ CHANGE ME: pick a neighborhood to investigate further
FOCUS_NEIGHBORHOOD = "Southside"  # try "Riverside", "Uptown", "Old Town", ...
# ▶ CHANGE ME: minimum number of requests a category needs to be shown
MIN_REQUESTS = 15

subset = df311[df311["neighborhood"] == FOCUS_NEIGHBORHOOD]
cat_counts_focus = subset["category"].value_counts()
cat_counts_focus = cat_counts_focus[cat_counts_focus >= MIN_REQUESTS]
print(f"Categories with at least {MIN_REQUESTS} requests in {FOCUS_NEIGHBORHOOD}:")
print(cat_counts_focus)

## Artifact: AI Use Log entry #1
This is your first entry in the mandatory AI Use Log required by the course's Responsible AI policy.

In [ ]:
ai_use_log_entry = pd.DataFrame([{
    "log_entry": 1,
    "tool": "Simulated chat-AI (Track B) -- substitute your actual approved chat tool if you used Track A",
    "date": datetime.today().strftime("%Y-%m-%d"),
    "prompt": prompt,
    "output_used_for": "Drafting a 1-paragraph overview of 311 request patterns for a planning memo",
    "human_verification_needed": (
        "Verify the top-category and slowest-neighborhood claims against the raw counts; "
        "check that no neighborhood was silently excluded; confirm the date range is correct."
    ),
    "human_edits_made": "Added neighborhood-level response-time context that the AI summary omitted.",
}])
print(ai_use_log_entry.to_string(index=False))
ai_use_log_entry.to_csv("ai_use_log_entry_1.csv", index=False)
print("\nSaved artifact: ai_use_log_entry_1.csv")

## Reflect (answer in your own words — this is part of your mini-task)

1. What did the tool assume?
2. Who is missing from this data?
3. What would change your recommendation?
4. What must a human verify before this is used?
